# Chapter 1 · The OTel RAG pipeline — OTel as a self-healing retrieval brain

_How Open Telco models become the **retrieval brain** of a self-healing network — the full grounding pipeline on Databricks._

In [Step 0](./00_starter_load_and_inference.ipynb) you loaded one model. Here we assemble the whole pipeline — the pattern a production Databricks self-healing NOC uses to ground its answers:

```
question -> EMBED -> RETRIEVE (vector search) -> RERANK -> GROUND (cite) -> ABSTAIN?
            335M     cosine (normalized!)        0.6B      top-K + sources   safety
```

**Production lessons baked in (learned the hard way):**
- **Embed:** the model returns raw vectors — **L2-normalize** corpus *and* query. Databricks Vector Search ranks by L2 only, so normalizing makes L2 == cosine.
- **Retrieve:** raw query text, no prefix (this fine-tune is prefix-insensitive).
- **Rerank (`OTel-Reranker-0.6B`):** only worth it if it **discriminates** — in production it sometimes returned near-constant scores, so we **validate before trusting** (a cell checks the spread).
- **Ground + abstain:** below a score threshold, **abstain** instead of hallucinating — the safety gate a self-healing loop needs.

In [Chapter 2](./02_self_healing_agent_loop.ipynb) this becomes the agent's `retrieve_standards`. First run downloads embedding (~335M) + reranker (~0.6B); the reranker needs `trust_remote_code=True`. (OTel = Open Telco.)


In [0]:
# Databricks: run this first — installs deps, then restarts Python.
# (The ML runtime pre-ships torch + transformers; serverless does not.)
%pip install -q "sentence-transformers>=3.0.0" "transformers>=4.40.0"
try:
    dbutils.library.restartPython()
except NameError:
    pass  # local Jupyter: `pip install -r ../requirements.txt` in a terminal instead

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

EMB_ID     = "farbodtavakkoli/OTel-Embedding-335M"   # BGE-large based, ~335M, CPU-fine
RERANK_ID  = "farbodtavakkoli/OTel-Reranker-0.6B"    # Qwen3-0.6B cross-encoder
print("imports ok")

imports ok


## 1. A tiny telecom corpus

A handful of 3GPP / O-RAN / ops-playbook snippets — the same knowledge the [`app/app.py`](../app/app.py) self-healing agent cites. In production this is hundreds of thousands of passages in a Databricks Vector Search index; here it's an in-memory list so you can see every moving part.

In [0]:
CORPUS = [
    {"cite": "3GPP TS 38.214 5.2", "text": "Low SINR/CQI forces a lower MCS, reducing per-UE throughput even at moderate PRB. Low throughput with LOW PRB load indicates a radio-quality (interference/coverage) problem, not congestion."},
    {"cite": "3GPP TS 36.213 7.2", "text": "LTE downlink throughput saturates as PRB utilization approaches 100%. Sustained PRB above 90% across neighboring cells in the busy hour is the signature of CONGESTION (capacity limit), not radio quality."},
    {"cite": "O-RAN WG1 UC",      "text": "Congestion remediation order: (1) load-balance/traffic-steer to under-utilized neighbors, (2) enable/verify carrier aggregation, (3) add carrier/spectrum, (4) cell split or new site."},
    {"cite": "3GPP TS 36.331 8.1", "text": "PCI collision between neighbor cells corrupts measurement reports and handovers, degrading SINR and raising drop and handover-failure rates. Resolve PCI conflicts before RF optimization."},
    {"cite": "RF Ops Playbook",    "text": "Correlate recurring EXTERNAL_INTERFERENCE_UL alarms with low-SINR cells before adjusting antenna tilt or transmit power."},
    {"cite": "O-RAN F1",           "text": "The F1 interface connects the O-RAN Distributed Unit (O-DU) to the O-RAN Central Unit (O-CU), carrying control-plane (F1-C) and user-plane (F1-U) traffic."},
    {"cite": "CPRI/eCPRI Ops",     "text": "Loss of the CPRI/eCPRI fronthaul link between the radio unit and the baseband unit takes the cell off the air; a technician dispatch to inspect the fiber path is required."},
    {"cite": "TM Forum Open API",  "text": "Standardized management interfaces enable vendor-agnostic data collection across Huawei/Ericsson/Nokia OSS/BSS for closed-loop automation."},
]
texts = [d["text"] for d in CORPUS]
print(f"{len(CORPUS)} passages")

8 passages


## 2. Embed + index — and the normalization lesson

**The lesson:** the embedder returns raw (unnormalized) vectors. We L2-normalize them so that:
- cosine similarity becomes a plain dot product, and
- crucially, **Databricks Vector Search ranks by L2 distance only** (HNSW) — with unit vectors, L2 ranking is identical to cosine ranking. Normalizing here is what makes this demo behave like the production index.

`sentence-transformers` gives us `normalize_embeddings=True` for free; we do the same to the query later.

In [0]:
embedder = SentenceTransformer(EMB_ID)
print(f"loaded {EMB_ID}  (dim={embedder.get_sentence_embedding_dimension()})")

# The 'index': a matrix of L2-normalized passage vectors (one per corpus row).
index = embedder.encode(texts, normalize_embeddings=True)
print("index shape:", index.shape, "| row norms ~", np.round(np.linalg.norm(index, axis=1)[:3], 3))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/469 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Default prompt name is set to 'Retrieval-query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.
/home/spark-526cd8cc-88ab-4015-ad02-c1/.ipykernel/92/command-7431061339600648-3394823560:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"loaded {EMB_ID}  (dim={embedder.get_sentence_embedding_dimension()})")


loaded farbodtavakkoli/OTel-Embedding-335M  (dim=1024)
index shape: (8, 1024) | row norms ~ [1.002 1.001 1.   ]


## 3. Retrieve

Embed the query with the **same normalization**, then rank passages by cosine (= dot product with unit vectors). Use **raw query text, no prompt prefix** — this OTel fine-tune family is prefix-insensitive.

In [0]:
def retrieve(query, k=4):
    q = embedder.encode(query, normalize_embeddings=True)   # normalize the QUERY too
    scores = index @ q
    order = np.argsort(scores)[::-1][:k]
    return [{"cite": CORPUS[i]["cite"], "text": texts[i], "vec_score": float(scores[i])} for i in order]

for hit in retrieve("cells show low downlink throughput but PRB load is low and SINR is poor"):
    print(f"  [{hit['vec_score']:.3f}] {hit['cite']:<20} {hit['text'][:70]}...")

  [0.753] 3GPP TS 36.213 7.2   LTE downlink throughput saturates as PRB utilization approaches 100%. ...
  [0.700] 3GPP TS 38.214 5.2   Low SINR/CQI forces a lower MCS, reducing per-UE throughput even at mo...
  [0.495] RF Ops Playbook      Correlate recurring EXTERNAL_INTERFERENCE_UL alarms with low-SINR cell...
  [0.494] CPRI/eCPRI Ops       Loss of the CPRI/eCPRI fronthaul link between the radio unit and the b...


## 4. Rerank — with a reality check

A cross-encoder reads the (query, passage) pair *together* and scores relevance directly — usually sharper than the bi-encoder vector score. We load `OTel-Reranker-0.6B` (`trust_remote_code=True`; Qwen3-based).

**The lesson:** a reranker is only worth its latency if it actually *discriminates*. In production this exact reranker sometimes returned near-constant scores and had to be disabled. So section 4b **validates** it before we trust it.

In [0]:
rk_tok = AutoTokenizer.from_pretrained(RERANK_ID, trust_remote_code=True)
rk_model = AutoModelForSequenceClassification.from_pretrained(RERANK_ID, trust_remote_code=True)
rk_model.eval()
print(f"loaded {RERANK_ID}")

@torch.no_grad()
def rerank_score(query, passage):
    inputs = rk_tok([[query, passage]], padding=True, truncation=True, return_tensors="pt")
    logits = rk_model(**inputs).logits
    return float(logits.flatten()[-1])   # single relevance logit; higher = more relevant

def rerank(query, hits):
    for h in hits:
        h["rerank_score"] = rerank_score(query, h["text"])
    return sorted(hits, key=lambda h: h["rerank_score"], reverse=True)

config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

loaded farbodtavakkoli/OTel-Reranker-0.6B


### 4b. Validate the reranker discriminates

Score one query against an obviously-relevant vs an obviously-irrelevant passage, and check the spread across the whole corpus. If the scores are flat, **don't trust the reranker** — fall back to vector order (exactly what the production demo did).

In [0]:
probe_q   = "two adjacent cells were configured with the same PCI"
relevant  = next(d["text"] for d in CORPUS if d["cite"] == "3GPP TS 36.331 8.1")   # PCI collision
irrelevant = next(d["text"] for d in CORPUS if d["cite"] == "TM Forum Open API")   # unrelated

s_rel, s_irr = rerank_score(probe_q, relevant), rerank_score(probe_q, irrelevant)
all_scores = [rerank_score(probe_q, t) for t in texts]
spread = max(all_scores) - min(all_scores)

print(f"relevant  : {s_rel:+.3f}")
print(f"irrelevant: {s_irr:+.3f}")
print(f"corpus score spread: {spread:.3f}")

RERANKER_OK = (s_rel > s_irr) and (spread > 0.1)
print("\nVERDICT:", "PASS - reranker discriminates, use it" if RERANKER_OK
      else "WARN - scores nearly flat; fall back to vector order")

relevant  : -3.125
irrelevant: -3.531
corpus score spread: 2.594

VERDICT: PASS - reranker discriminates, use it


## 5. Ground + abstain — the safety gate

Assemble a cited grounding block from the top passages. **If the best relevance is below threshold, abstain** — return "insufficient grounded context" instead of guessing. A self-healing loop must never recommend touching a live network on weak evidence.

In [0]:
ABSTAIN_BELOW = 0.35   # min vector score of the top hit to be willing to answer

def ground(query, k=4, top_n=2):
    hits = retrieve(query, k=k)
    if not hits or hits[0]["vec_score"] < ABSTAIN_BELOW:
        return {"answered": False, "reason": f"insufficient grounded context (top score {hits[0]['vec_score']:.3f} < {ABSTAIN_BELOW})", "sources": []}
    ranked = rerank(query, hits) if RERANKER_OK else hits    # honor the validation verdict
    chosen = ranked[:top_n]
    grounding = "\n".join(f"  - [{h['cite']}] {h['text']}" for h in chosen)
    return {"answered": True, "stage": "reranked" if RERANKER_OK else "vector-only",
            "grounding": grounding, "sources": [h["cite"] for h in chosen]}

## 6. Run the full pipeline on self-healing scenarios

The same two incidents the `app/app.py` agent handles — a radio-quality fault vs. a congestion fault — plus one deliberately off-domain question to watch the **abstain** gate fire.

In [0]:
def run(query):
    print(f"\nQ: {query}")
    out = ground(query)
    if not out["answered"]:
        print(f"   ABSTAIN -> {out['reason']}")
        return
    print(f"   grounded ({out['stage']}) on: {', '.join(out['sources'])}")
    print(out["grounding"])

run("cells show low downlink throughput but PRB load is low and SINR is poor")
run("every neighboring cell is at 98% PRB during the evening busy hour and users are slow")
run("what is the recommended cooking temperature for a medium-rare steak")   # should abstain


Q: cells show low downlink throughput but PRB load is low and SINR is poor
   grounded (reranked) on: 3GPP TS 38.214 5.2, 3GPP TS 36.213 7.2
  - [3GPP TS 38.214 5.2] Low SINR/CQI forces a lower MCS, reducing per-UE throughput even at moderate PRB. Low throughput with LOW PRB load indicates a radio-quality (interference/coverage) problem, not congestion.
  - [3GPP TS 36.213 7.2] LTE downlink throughput saturates as PRB utilization approaches 100%. Sustained PRB above 90% across neighboring cells in the busy hour is the signature of CONGESTION (capacity limit), not radio quality.

Q: every neighboring cell is at 98% PRB during the evening busy hour and users are slow
   grounded (reranked) on: 3GPP TS 36.213 7.2, 3GPP TS 36.331 8.1
  - [3GPP TS 36.213 7.2] LTE downlink throughput saturates as PRB utilization approaches 100%. Sustained PRB above 90% across neighboring cells in the busy hour is the signature of CONGESTION (capacity limit), not radio quality.
  - [3GPP TS 36.331 8.1] PCI c

## What you just built

✅ The complete OTel grounding pipeline — **embed → retrieve → rerank → ground → abstain** — on Databricks.  
✅ With production lessons built in: **L2-normalized** vectors, a **validated** reranker, and a **safety gate** that abstains on weak evidence.

This is exactly the `retrieve_standards` capability the self-healing agent needs — real, cited, and honest.

**Chapter 2** wires this pipeline into the ReAct loop of [`app/app.py`](../app/app.py), so the agent's Think -> Act -> Observe -> Reflect cycle grounds every step on real OTel retrieval instead of a keyword match.

**Chapters 3-4** take this exact pipeline to Databricks: register the models in Unity Catalog, serve them (embedding on CPU/GPU serving; reranker + LLM on GPU / provisioned throughput), build the Vector Search index at scale, and — the part most demos skip — **capture every inference** in inference tables with monitoring and a cost-economics ledger.

_See [`./01_otel_rag_pipeline.ipynb`](./01_otel_rag_pipeline.ipynb) and the [roadmap](./00_starter_load_and_inference.ipynb)._